[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/Alternative_Role_Classifier_Analyzer.ipynb)

**📝 Before using:** Update the GitHub URL above with your actual username and repository name.

# 🔍 Alternative Role Classification Analyzer

**Identify plausible alternative role classifications based on job descriptions**

This notebook analyzes job classifications to identify **other reasonable role classifications** that could have been assigned based on the job description, regardless of whether the original classification was "correct."

## Key Features:
- **Multi-Model Support**: Switch between Claude, Gemini, Llama, Qwen, and DeepSeek models
- **Alternative Focus**: Identifies plausible alternatives, not accuracy scores
- **Context-Aware**: Uses KSACs, ground truth, and problem role patterns
- **Easy Configuration**: Simple model switching without code changes

## Required Input Files:
Upload `Evaluation Resources.zip` containing:
- **Primary Inputs:**
  - `Sample_JDs.json` – Original job descriptions
  - `Job_Classifications_Batch.json` – Classifier output
- **Reference Resources:**
  - `Ground_Truth_Masterfile.json` – Known role mappings
  - `MNPS_KSACs.json` – Role competency data
  - `Problem_Role_Cheatsheet.json` – Known ambiguity patterns

## Output:
A CSV table with:
- Job Code
- Classified Role (from classifier)
- Other Plausible Roles (alternatives identified)
- Analysis Reasoning (why alternatives are plausible)

In [ ]:
# =========================================================================
# 🚨 STEP 1: CHOOSE YOUR MODEL MODE
# =========================================================================

# Select one of the following modes:
# 
# API-based (Requires API Key in Colab Secrets):
#   1. 'CLAUDE_API'            - Anthropic Claude Sonnet 4 / 4.5
#   2. 'GEMINI_API'            - Google Gemini 2.0 / 2.5
#
# Local/HuggingFace (Requires Colab GPU):
#   3. 'LLAMA_8B'              - Meta Llama 3.1 8B Instruct
#   4. 'LLAMA_70B'             - Meta Llama 3.1 70B Instruct (A100 recommended)
#   5. 'QWEN_7B'               - Qwen 2.5 7B Instruct
#   6. 'QWEN_80B'              - Qwen 3 Next 80B Instruct (A100 required)
#   7. 'DEEPSEEK_8B_QWEN'      - DeepSeek R1 Distill Qwen 8B
#   8. 'DEEPSEEK_8B_LLAMA'     - DeepSeek R1 Distill Llama 8B

MODEL_MODE = 'CLAUDE_API'  # <<< CHANGE THIS TO SWITCH MODELS >>>

# =========================================================================
# 🔑 STEP 2: API Keys & General Configuration
# =========================================================================

# API Keys (Set via Colab Secrets 🔑 or manually here)
ANTHROPIC_API_KEY = ""
GEMINI_API_KEY = ""

# General Settings
ZIP_FILE_PATH = "/content/Evaluation Resources.zip"  # Expected location after upload
OUTPUT_PATH = "/content/analysis_results"
MAX_TOKENS = 2000
TEMPERATURE = 0.0  # Use 0.0 for deterministic, analytical reasoning
MAX_RETRIES = 3
RETRY_DELAY = 2.0

# CLAUDE Specific Settings (only used if MODEL_MODE = 'CLAUDE_API')
CLAUDE_MODEL_NAME = "claude-sonnet-4-20250514"  # Sonnet 4 is powerful and cost-effective
CLAUDE_FALLBACK_MODEL = "claude-sonnet-4-5-20250929"  # Fallback to Sonnet 4.5 if needed
ENABLE_PROMPT_CACHING = True  # Highly recommended for cost savings

# GEMINI Specific Settings (only used if MODEL_MODE = 'GEMINI_API')
GEMINI_MODEL_NAME = "gemini-2.0-flash-exp"  # Fast and cost-effective

# LLAMA Specific Settings (only used if MODEL_MODE = 'LLAMA_8B' or 'LLAMA_70B')
LLAMA_8B_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
LLAMA_70B_MODEL_NAME = "meta-llama/Llama-3.1-70B-Instruct"
LLAMA_USE_4BIT = True  # Recommended to save VRAM

# QWEN Specific Settings (only used if MODEL_MODE = 'QWEN_7B' or 'QWEN_80B')
QWEN_7B_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
QWEN_80B_MODEL_NAME = "Qwen/Qwen3-Next-80B-A3B-Instruct"
QWEN_USE_4BIT = True  # Recommended to save VRAM

# DEEPSEEK Specific Settings (only used if MODEL_MODE starts with 'DEEPSEEK')
DEEPSEEK_QWEN_MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-8B"
DEEPSEEK_LLAMA_MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
DEEPSEEK_USE_4BIT = True  # Recommended to save VRAM

# =========================================================================
# 🔄 STEP 3: Dependency Installation (Runs automatically)
# =========================================================================

import os
import json
import zipfile
import time
from typing import Dict, List, Optional
import pandas as pd
from tqdm.auto import tqdm
from google.colab import files, userdata

print(f"\n{'='*70}")
print(f"Selected Model Mode: {MODEL_MODE}")
print(f"{'='*70}\n")

# Build installation command based on selected model
install_command = "pip install -q pandas numpy matplotlib seaborn plotly networkx tqdm"

if MODEL_MODE == 'CLAUDE_API':
    install_command += " anthropic>=0.34.0"
    try:
        ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
        print("✅ Anthropic API key loaded from Colab secrets.")
    except:
        print("⚠️ ANTHROPIC_API_KEY not found in secrets. Please set it manually above.")

elif MODEL_MODE == 'GEMINI_API':
    install_command += " google-genai"
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
        print("✅ Gemini API key loaded from Colab secrets.")
    except:
        print("⚠️ GEMINI_API_KEY not found in secrets. Please set it manually above.")

elif MODEL_MODE in ['LLAMA_8B', 'LLAMA_70B', 'QWEN_7B', 'QWEN_80B', 'DEEPSEEK_8B_QWEN', 'DEEPSEEK_8B_LLAMA']:
    install_command += " torch torchvision torchaudio transformers accelerate bitsandbytes"
    print("⚠️ Local model selected. Ensure you have GPU runtime enabled.")
    print("   Runtime → Change runtime type → Hardware accelerator: GPU")
    if MODEL_MODE in ['LLAMA_70B', 'QWEN_80B']:
        print("   ⚠️ Large model selected - A100 GPU strongly recommended.")

else:
    raise ValueError(f"Invalid MODEL_MODE: {MODEL_MODE}. Choose from: CLAUDE_API, GEMINI_API, LLAMA_8B, LLAMA_70B, QWEN_7B, QWEN_80B, DEEPSEEK_8B_QWEN, DEEPSEEK_8B_LLAMA")

print("\n📦 Installing dependencies...")
os.system(install_command)
print("✅ Dependencies installed.\n")

## 2. File Upload and Data Loading

In [ ]:
print("⬆️ Please upload 'Evaluation Resources.zip' when prompted.\n")
uploaded = files.upload()

if 'Evaluation Resources.zip' not in uploaded:
    raise FileNotFoundError("❌ 'Evaluation Resources.zip' not found. Please upload the file.")

# Extract zip file
with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
    zip_ref.extractall("/content/")
print("✅ Files extracted successfully.\n")

# =========================================================================
# 📂 LOAD DATA FILES
# =========================================================================

def find_and_load_json(filename: str) -> List[Dict]:
    """Search for and load a JSON file from extracted contents."""
    for root, dirs, files_list in os.walk("/content/"):
        for file in files_list:
            if file.lower() == filename.lower():
                filepath = os.path.join(root, file)
                with open(filepath, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                print(f"  ✅ Loaded {filename}: {len(data)} records")
                return data
    print(f"  ❌ Warning: {filename} not found")
    return []

print("🔍 Loading data files...\n")

# Load all required files
sample_jds = find_and_load_json('Sample_JDs.json')
classifications = find_and_load_json('Job_Classifications_Batch.json')
ground_truth = find_and_load_json('Ground_Truth_Masterfile.json')
ksacs = find_and_load_json('MNPS_KSACs.json')
problem_roles = find_and_load_json('Problem_Role_Cheatsheet.json')

# Store in dictionary for easy access
datasets = {
    'sample_jds': sample_jds,
    'classifications': classifications,
    'ground_truth': ground_truth,
    'ksacs': ksacs,
    'problem_roles': problem_roles
}

print(f"\n✅ Data loading complete!")

## 3. Data Preparation

In [ ]:
# =========================================================================
# 🔗 MERGE JOB DESCRIPTIONS WITH CLASSIFICATIONS
# =========================================================================

def prepare_analysis_data(datasets: Dict) -> pd.DataFrame:
    """Merge job descriptions with classifications for analysis."""
    
    # Convert to DataFrames
    df_jds = pd.DataFrame(datasets['sample_jds'])
    df_class = pd.DataFrame(datasets['classifications'])
    
    # Standardize job titles for merging
    df_jds['Job Title'] = df_jds['Job Title'].astype(str).str.strip()
    df_class['job_title_original'] = df_class['job_title_original'].astype(str).str.strip()
    
    # Merge on job title (1:1 relationship)
    df_merged = pd.merge(
        df_class,
        df_jds,
        left_on='job_title_original',
        right_on='Job Title',
        how='inner'
    )
    
    # Create full job description text
    def create_jd_text(row):
        parts = []
        if 'Position Summary' in row and pd.notna(row['Position Summary']):
            parts.append(f"Position Summary: {row['Position Summary']}")
        if 'Essential Functions' in row and pd.notna(row['Essential Functions']):
            parts.append(f"Essential Functions: {row['Essential Functions']}")
        if 'Education' in row and pd.notna(row['Education']):
            parts.append(f"Education: {row['Education']}")
        if 'Work Experience' in row and pd.notna(row['Work Experience']):
            parts.append(f"Work Experience: {row['Work Experience']}")
        if 'Knowledge, Skills and Abilities' in row and pd.notna(row['Knowledge, Skills and Abilities']):
            parts.append(f"Knowledge, Skills and Abilities: {row['Knowledge, Skills and Abilities']}")
        return "\n\n".join(parts)
    
    df_merged['full_job_description'] = df_merged.apply(create_jd_text, axis=1)
    
    print(f"✅ Prepared {len(df_merged)} job records for analysis")
    
    if len(df_merged) != len(df_class):
        print(f"⚠️ Warning: {len(df_class) - len(df_merged)} records did not merge")
    
    return df_merged

# Prepare the analysis dataset
df_analysis = prepare_analysis_data(datasets)

# Display sample
print("\n📋 Sample records:")
display(df_analysis[['Job Code', 'job_title_original', 'major_role_group']].head())

## 4. Model Initialization

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# =========================================================================
# 🤖 MODEL INITIALIZATION
# =========================================================================

model_client = None
model_tokenizer = None
model_name = None

# Determine model name based on MODE
if MODEL_MODE == 'CLAUDE_API':
    model_name = CLAUDE_MODEL_NAME
elif MODEL_MODE == 'GEMINI_API':
    model_name = GEMINI_MODEL_NAME
elif MODEL_MODE == 'LLAMA_8B':
    model_name = LLAMA_8B_MODEL_NAME
elif MODEL_MODE == 'LLAMA_70B':
    model_name = LLAMA_70B_MODEL_NAME
elif MODEL_MODE == 'QWEN_7B':
    model_name = QWEN_7B_MODEL_NAME
elif MODEL_MODE == 'QWEN_80B':
    model_name = QWEN_80B_MODEL_NAME
elif MODEL_MODE == 'DEEPSEEK_8B_QWEN':
    model_name = DEEPSEEK_QWEN_MODEL_NAME
elif MODEL_MODE == 'DEEPSEEK_8B_LLAMA':
    model_name = DEEPSEEK_LLAMA_MODEL_NAME

print(f"\n{'='*70}")
print(f"Initializing: {MODEL_MODE}")
print(f"Model: {model_name}")
print(f"{'='*70}\n")

def create_reference_context() -> str:
    """Create reference context from KSAC and problem role data."""
    context_parts = []
    
    # Add KSAC reference (truncated for context limits)
    if datasets['ksacs']:
        ksac_summary = json.dumps(datasets['ksacs'][:50], indent=2)  # First 50 roles
        context_parts.append(f"**KSAC Role Reference (Sample):**\n{ksac_summary}")
    
    # Add problem role patterns
    if datasets['problem_roles']:
        problem_summary = json.dumps(datasets['problem_roles'], indent=2)
        context_parts.append(f"**Known Ambiguity Patterns:**\n{problem_summary}")
    
    return "\n\n".join(context_parts)

reference_context = create_reference_context()

# Initialize based on model type
if MODEL_MODE == 'CLAUDE_API':
    from anthropic import Anthropic
    model_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    print(f"✅ Claude API client initialized")

elif MODEL_MODE == 'GEMINI_API':
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model_client = genai.GenerativeModel(model_name=model_name)
    print(f"✅ Gemini API client initialized")

else:  # Local models
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    
    print(f"⏳ Loading {model_name}...")
    print("   This may take several minutes for large models.\n")
    
    # Determine if using 4-bit quantization
    use_4bit = False
    if MODEL_MODE in ['LLAMA_8B', 'LLAMA_70B']:
        use_4bit = LLAMA_USE_4BIT
    elif MODEL_MODE in ['QWEN_7B', 'QWEN_80B']:
        use_4bit = QWEN_USE_4BIT
    elif MODEL_MODE in ['DEEPSEEK_8B_QWEN', 'DEEPSEEK_8B_LLAMA']:
        use_4bit = DEEPSEEK_USE_4BIT
    
    if use_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model_client = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
    else:
        model_client = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
    
    model_tokenizer = AutoTokenizer.from_pretrained(model_name)
    if model_tokenizer.pad_token is None:
        model_tokenizer.pad_token = model_tokenizer.eos_token
    
    print(f"✅ Model loaded successfully")
    if torch.cuda.is_available():
        print(f"   GPU Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB\n")

print("🎯 Model ready for analysis!")

## 5. Alternative Role Analyzer

In [ ]:
import time
from tqdm.auto import tqdm

# =========================================================================
# 🔍 ALTERNATIVE ROLE ANALYZER
# =========================================================================

class AlternativeRoleAnalyzer:
    def __init__(self, df, datasets, model_mode, client, model_name_str, tokenizer=None, reference_ctx=None):
        self.df = df
        self.datasets = datasets
        self.model_mode = model_mode
        self.client = client
        self.model_name = model_name_str
        self.tokenizer = tokenizer
        self.reference_context = reference_ctx or ""
    
    def create_analysis_prompt(self, row: pd.Series) -> str:
        """Create prompt for alternative role identification."""
        
        job_code = row.get('Job Code', 'N/A')
        job_title = row.get('job_title_original', 'N/A')
        classified_role = row.get('major_role_group', 'N/A')
        job_description = row.get('full_job_description', 'N/A')
        
        # Get ground truth if available
        ground_truth = 'N/A'
        for gt_record in self.datasets['ground_truth']:
            if gt_record.get('Job Title', '').strip() == job_title.strip():
                ground_truth = gt_record.get('major_role_group', 'N/A')
                break
        
        prompt = f"""# ALTERNATIVE ROLE IDENTIFICATION TASK

You are an HR classification expert. Analyze the job description below and identify **other plausible major_role_group classifications** that could reasonably fit this position.

**Important**: This is NOT about determining if the classifier was correct or incorrect. Instead, identify alternative roles that would also make sense based on:
- KSAC (Knowledge, Skills, Abilities, Competencies) overlap
- Role scope and responsibilities
- Known ambiguity patterns between similar roles

## JOB INFORMATION

**Job Code:** {job_code}
**Job Title:** {job_title}
**Classified Role:** {classified_role}
**Ground Truth Role:** {ground_truth}

**Job Description:**
{job_description}

## REFERENCE CONTEXT

{self.reference_context}

## YOUR TASK

1. Review the job description carefully
2. Consider KSAC patterns and role similarities
3. Identify 1-5 alternative major_role_group values that could plausibly fit
4. Explain why each alternative is reasonable

## OUTPUT FORMAT

Return ONLY valid JSON with this structure:

```json
{{
  "job_code": "{job_code}",
  "classified_role": "{classified_role}",
  "other_plausible_roles": ["Role1", "Role2", "Role3"],
  "reasoning": "Brief explanation of why these alternatives are plausible based on KSAC overlap, role scope, or known ambiguity patterns."
}}
```

If no plausible alternatives exist, return an empty list for other_plausible_roles.
"""
        return prompt
    
    def generate_response(self, prompt: str) -> Dict:
        """Generate analysis using configured model."""
        
        for attempt in range(MAX_RETRIES):
            try:
                if self.model_mode == 'CLAUDE_API':
                    response = self.client.messages.create(
                        model=self.model_name,
                        max_tokens=MAX_TOKENS,
                        temperature=TEMPERATURE,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    response_text = response.content[0].text
                
                elif self.model_mode == 'GEMINI_API':
                    import google.generativeai as genai
                    generation_config = genai.types.GenerationConfig(
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_TOKENS
                    )
                    response = self.client.generate_content(
                        prompt,
                        generation_config=generation_config
                    )
                    response_text = response.text
                
                else:  # Local models
                    import torch
                    messages = [
                        {"role": "system", "content": "You are an HR classification expert."},
                        {"role": "user", "content": prompt}
                    ]
                    
                    formatted_prompt = self.tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True
                    )
                    
                    inputs = self.tokenizer(
                        formatted_prompt,
                        return_tensors="pt",
                        truncation=True,
                        max_length=4096
                    ).to(self.client.device)
                    
                    with torch.no_grad():
                        outputs = self.client.generate(
                            **inputs,
                            max_new_tokens=MAX_TOKENS,
                            temperature=TEMPERATURE,
                            do_sample=TEMPERATURE > 0
                        )
                    
                    response_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                    # Extract assistant response
                    if "assistant" in response_text.lower():
                        response_text = response_text.split("assistant")[-1].strip()
                
                # Parse JSON response
                response_text = response_text.strip()
                if response_text.startswith('```json'):
                    response_text = response_text[7:]
                if response_text.startswith('```'):
                    response_text = response_text[3:]
                if response_text.endswith('```'):
                    response_text = response_text[:-3]
                response_text = response_text.strip()
                
                result = json.loads(response_text)
                result['status'] = 'success'
                return result
            
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = RETRY_DELAY * (2 ** attempt)
                    print(f"\r⚠️ Retry {attempt + 1}/{MAX_RETRIES} after {wait_time:.0f}s...", end="")
                    time.sleep(wait_time)
                else:
                    return {
                        'job_code': 'N/A',
                        'classified_role': 'N/A',
                        'other_plausible_roles': [],
                        'reasoning': f'Analysis failed: {str(e)[:200]}',
                        'status': 'failed'
                    }
    
    def analyze_all(self) -> pd.DataFrame:
        """Analyze all job records."""
        results = []
        
        print(f"\n🚀 Starting analysis with {self.model_mode} ({self.model_name})...\n")
        
        for idx, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Analyzing jobs"):
            prompt = self.create_analysis_prompt(row)
            result = self.generate_response(prompt)
            
            results.append({
                'Job Code': row.get('Job Code', 'N/A'),
                'Classified Role': row.get('major_role_group', 'N/A'),
                'Other Plausible Roles': ', '.join(result.get('other_plausible_roles', [])),
                'Analysis Reasoning': result.get('reasoning', 'N/A'),
                'Status': result.get('status', 'failed')
            })
        
        return pd.DataFrame(results)

# Initialize analyzer
analyzer = AlternativeRoleAnalyzer(
    df=df_analysis,
    datasets=datasets,
    model_mode=MODEL_MODE,
    client=model_client,
    model_name_str=model_name,
    tokenizer=model_tokenizer,
    reference_ctx=reference_context
)

print("✅ Analyzer ready!")

## 6. Run Analysis

In [ ]:
# =========================================================================
# ▶️ EXECUTE ANALYSIS
# =========================================================================

df_results = analyzer.analyze_all()

# Calculate statistics
success_rate = (df_results['Status'] == 'success').mean()
jobs_with_alternatives = (df_results['Other Plausible Roles'].str.len() > 0).sum()
total_jobs = len(df_results)

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE")
print("="*70)
print(f"Total Jobs Analyzed: {total_jobs}")
print(f"Successful Analyses: {(df_results['Status'] == 'success').sum()} ({success_rate:.1%})")
print(f"Jobs with Alternatives: {jobs_with_alternatives} ({jobs_with_alternatives/total_jobs:.1%})")
print("="*70 + "\n")

# Display results
print("📋 Sample Results:\n")
display(df_results.drop(columns=['Status']).head(10))

## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("📊 Generating visualizations...\n")

# =========================================================================
# 1. Distribution of Alternative Counts
# =========================================================================

df_results['alt_count'] = df_results['Other Plausible Roles'].apply(
    lambda x: len([r.strip() for r in str(x).split(',') if r.strip()])
)

plt.figure(figsize=(10, 6))
sns.countplot(data=df_results, x='alt_count', palette='viridis')
plt.title('Distribution of Alternative Role Counts', fontsize=16, fontweight='bold')
plt.xlabel('Number of Alternative Roles Identified', fontsize=12)
plt.ylabel('Number of Jobs', fontsize=12)
plt.xticks(range(0, df_results['alt_count'].max() + 1))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/alternative_distribution.png", dpi=300)
plt.show()

print("  ✅ Alternative distribution chart saved")

# =========================================================================
# 2. Top Classified Roles
# =========================================================================

plt.figure(figsize=(12, 6))
top_roles = df_results['Classified Role'].value_counts().head(10)
sns.barplot(x=top_roles.values, y=top_roles.index, palette='rocket')
plt.title('Top 10 Most Common Classified Roles', fontsize=16, fontweight='bold')
plt.xlabel('Number of Jobs', fontsize=12)
plt.ylabel('Role', fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/top_roles.png", dpi=300)
plt.show()

print("  ✅ Top roles chart saved")

# =========================================================================
# 3. Analysis Status
# =========================================================================

plt.figure(figsize=(8, 8))
status_counts = df_results['Status'].value_counts()
colors = ['#2ecc71', '#e74c3c']
plt.pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
        colors=colors[:len(status_counts)], startangle=90)
plt.title('Analysis Success Rate', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/status_breakdown.png", dpi=300)
plt.show()

print("  ✅ Status breakdown chart saved\n")

## 8. Export Results

In [ ]:
import shutil

# =========================================================================
# 💾 EXPORT RESULTS
# =========================================================================

# Save main results CSV
output_csv = f"{OUTPUT_PATH}/alternative_roles_analysis_{MODEL_MODE.lower()}.csv"
df_results.drop(columns=['Status', 'alt_count']).to_csv(output_csv, index=False)
print(f"✅ Results saved to: {output_csv}\n")

# Save summary statistics
summary = {
    'model_used': MODEL_MODE,
    'model_name': model_name,
    'total_jobs': total_jobs,
    'successful_analyses': int((df_results['Status'] == 'success').sum()),
    'success_rate': f"{success_rate:.1%}",
    'jobs_with_alternatives': int(jobs_with_alternatives),
    'percentage_with_alternatives': f"{jobs_with_alternatives/total_jobs:.1%}",
    'avg_alternatives_per_job': float(df_results['alt_count'].mean())
}

with open(f"{OUTPUT_PATH}/analysis_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print("📊 Analysis Summary:")
print(json.dumps(summary, indent=2))
print()

# Create downloadable zip
print("📦 Creating download package...")
shutil.make_archive('/content/analysis_results', 'zip', OUTPUT_PATH)

try:
    files.download('/content/analysis_results.zip')
    print("✅ Download initiated!\n")
except:
    print(f"⚠️ Could not auto-download. Files saved to: {OUTPUT_PATH}\n")

print("="*70)
print("🎉 ANALYSIS PIPELINE COMPLETE!")
print("="*70)

## 9. Model Selection Guide

### Quick Reference

| Model | Type | Speed | Cost | Memory | Best For |
|:------|:-----|:------|:-----|:-------|:---------|
| **Claude Sonnet 4** | API | ⚡⚡⚡ | 💰💰 | N/A | Production, accuracy |
| **Gemini 2.0 Flash** | API | ⚡⚡⚡ | 💰 | N/A | Fast, cost-effective |
| **Llama 3.1 8B** | Local | ⚡⚡ | Free | ~6GB | Quick testing |
| **Llama 3.1 70B** | Local | ⚡ | Free | ~35GB | High accuracy |
| **Qwen 2.5 7B** | Local | ⚡⚡ | Free | ~5GB | Efficient |
| **Qwen 3 80B** | Local | ⚡ | Free | ~40GB | Best local model |
| **DeepSeek-R1 8B** | Local | ⚡⚡ | Free | ~6GB | Reasoning tasks |

### API Models (Recommended)

**Setup:**
1. Get API key:
   - Claude: https://console.anthropic.com/
   - Gemini: https://aistudio.google.com/
2. Add to Colab Secrets (🔑 icon in left sidebar)
3. Set `MODEL_MODE = 'CLAUDE_API'` or `'GEMINI_API'` in Cell 1

**Pros:** Fast, no GPU needed, no memory limits  
**Cons:** Costs money per request

### Local Models

**Setup:**
1. Enable GPU: Runtime → Change runtime type → GPU
2. Choose model size based on available VRAM
3. Set `MODEL_MODE` to desired model in Cell 1

**Pros:** Free, unlimited requests, private  
**Cons:** Slower, requires GPU, memory constraints

### Recommendations

- **For testing/development:** Gemini 2.0 Flash (fast + cheap)
- **For production:** Claude Sonnet 4 (best accuracy)
- **For free/private:** Qwen 2.5 7B or Llama 3.1 8B
- **For best free results:** Qwen 3 80B or Llama 3.1 70B (requires A100 GPU)

### Troubleshooting

**Out of Memory (Local Models):**
- Use smaller model (8B instead of 70B)
- Ensure 4-bit quantization is enabled
- Request A100 GPU runtime

**API Errors:**
- Check API key in Colab Secrets
- Verify account has credits
- Check rate limits

**Slow Performance:**
- API models: Already optimized
- Local models: Use smaller model or API instead